In [1]:
%pip install -U openai pydantic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

In [4]:
PROJECT_DIR = Path("/Users/rejaimes/repos/coffee-project")
DATA_DIR = PROJECT_DIR / "data" / "processed"

load_dotenv("../src/.env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("API key not found. Check your .env file.")

client = OpenAI()

PROMPT_PATH = PROJECT_DIR / "src" / "prompt_v3.txt"
PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()

assert PROMPT, "Prompt file is empty."

MODEL = "gpt-4.1-mini"
PROMPT_VERSION = "full_v1_8qualifiers"

# Detect accidental prompt changes when resuming.
PROMPT_HASH = hashlib.sha256(PROMPT.encode("utf-8")).hexdigest()

In [5]:
class ReviewLabels(BaseModel):
    good_wifi: Literal[-1, 0, 1]
    easy_parking: Literal[-1, 0, 1]
    quiet_atmosphere: Literal[-1, 0, 1]
    enough_seating: Literal[-1, 0, 1]
    outlets_available: Literal[-1, 0, 1]
    work_friendly: Literal[-1, 0, 1]
    good_drinks: Literal[-1, 0, 1]
    fun_atmosphere: Literal[-1, 0, 1]


qualifiers = list(ReviewLabels.model_fields)

In [6]:
df_reviews = pd.read_csv(
    DATA_DIR / "santa_barbara_data.csv",
    low_memory=False,
)

df_run = df_reviews[
    ["review_id", "business_id", "text"]
].copy()

assert df_run["review_id"].notna().all(), "Missing review IDs."
assert df_run["review_id"].is_unique, "Duplicate review IDs."

blank_text = df_run["text"].fillna("").str.strip().eq("")

print("Total reviews:", len(df_run))
print("Reviews with blank text:", blank_text.sum())

# Keep blank-text reviews separate for inspection.
df_run.loc[blank_text].to_csv(
    DATA_DIR / "reviews_without_text.csv",
    index=False,
)

df_run = df_run.loc[~blank_text].copy()

print("Reviews eligible for labeling:", len(df_run))

Total reviews: 27128
Reviews with blank text: 0
Reviews eligible for labeling: 27128


In [ ]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = DATA_DIR / (
    f"llm_labels_{PROMPT_VERSION}_{run_timestamp}.jsonl"
)

with OUTPUT_PATH.open("x", encoding="utf-8") as file:
    for count, row in enumerate(
        df_run.to_dict("records"), start=1
    ):
        response = client.responses.parse(
            model=MODEL,
            input=[
                {"role": "system", "content": PROMPT},
                {"role": "user", "content": row["text"]},
            ],
            text_format=ReviewLabels,
            store=False,
        )

        if response.output_parsed is None:
            raise RuntimeError(
                f"No labels returned for {row['review_id']}."
            )

        record = {
            "review_id": row["review_id"],
            "business_id": row["business_id"],
            "labels": response.output_parsed.model_dump(),
            "requested_model": MODEL,
            "label_model": response.model,
            "prompt_version": PROMPT_VERSION,
            "prompt_hash": PROMPT_HASH,
            "label_date": datetime.now(timezone.utc).isoformat(),
        }

        file.write(json.dumps(record) + "\n")
        file.flush()

        print(f"Completed {count}/{len(df_run)}")

print("Finished. Results saved to:", OUTPUT_PATH)

Completed 1/27128
Completed 2/27128
Completed 3/27128
Completed 4/27128
Completed 5/27128
Completed 6/27128
Completed 7/27128
Completed 8/27128
Completed 9/27128
Completed 10/27128
Completed 11/27128
Completed 12/27128
Completed 13/27128
Completed 14/27128
Completed 15/27128
Completed 16/27128
Completed 17/27128
Completed 18/27128
Completed 19/27128
Completed 20/27128
Completed 21/27128
Completed 22/27128
Completed 23/27128
Completed 24/27128
Completed 25/27128
Completed 26/27128
Completed 27/27128
Completed 28/27128
Completed 29/27128
Completed 30/27128
Completed 31/27128
Completed 32/27128
Completed 33/27128
Completed 34/27128
Completed 35/27128
Completed 36/27128
Completed 37/27128
Completed 38/27128
Completed 39/27128
Completed 40/27128
Completed 41/27128
Completed 42/27128
Completed 43/27128
Completed 44/27128
Completed 45/27128
Completed 46/27128
Completed 47/27128
Completed 48/27128
Completed 49/27128
Completed 50/27128
Completed 51/27128
Completed 52/27128
Completed 53/27128
Co